# 1IEP order sensitivity at a fixed ROOT origin

Owned by DockingMT #17/#6. Six new searches keep the first ROOT ATOM line and origin fixed while reversing the other seven. Eighteen historical observations are authenticated by reference, with their original producers. All outcomes are retained. This notebook reads saved data; it does not rerun searches or select a ROOT policy.

In [1]:
from pathlib import Path
import gzip, hashlib, json, sys
import numpy as np
import pyunitwizard as puw
import dockingmt as dmt
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
path = root / 'devguide/validation/data/1iep_root_order/audit_2026-10-07.json.gz'
raw = gzip.decompress(path.read_bytes())
record = json.loads(raw)
assert sys.executable == record['interpreter']
assert sys.version_info[:2] == (3, 14)
assert len(record['new_runs']) == 6
baseline_path = root / record['baseline']['path']
assert hashlib.sha256(baseline_path.read_bytes()).hexdigest() == record['baseline']['sha256']
baseline = json.loads(gzip.decompress(baseline_path.read_bytes()))
print('New source base:', record['consumer_base_commit'])
print('Baseline published by:', record['baseline']['published_by'])
print('Original native producer:', record['baseline']['original_native_producer'])
print('Original JSON SHA-256:', hashlib.sha256(raw).hexdigest())
print('Fixed ROOT source atom/origin:', record['control']['first_root_source_atom_index'], record['control']['rigid_body_origin_angstrom'])
scored = record['fixed_conformation_score']
assert scored['scores'] == baseline['fixed_conformation_scores']['native']['scores']
assert dmt.verify_captured_inputs(scored['metadata']['scoring_history'][0]['backend_artifacts'])
print('Fixed-conformation score components:', scored['scores'])

New source base: dd571ef4dd0a1ddc9142e399a6da3683299a1db4
Baseline published by: dd571ef4dd0a1ddc9142e399a6da3683299a1db4
Original native producer: 7b11391e153f77f340fd039bc799352f90da83d3
Original JSON SHA-256: 587f7f89a8fe111faaa7135108e11dee151cc6f3a49707ebb2e52839b4fd4db0
Fixed ROOT source atom/origin: 28 [16.917, 46.907, 20.219]
Fixed-conformation score components: {'vina': -12.513, 'vina.flex_inter': 0.0, 'vina.flex_intra': 0.0, 'vina.lig_inter': -17.634, 'vina.lig_intra': -0.485, 'vina.lig_intra_best_pose': -0.485, 'vina.other_inter': 0.0, 'vina.torsions': 5.121}


In [2]:
print('variant                      effort seed poses first-heavy closest-heavy first-score')
for row in record['comparison_rows']:
    print(f"{row['variant']:30s} {row['exhaustiveness']:3d} {row['seed']:4d} {row['poses']:5d} {row['first_heavy_rmsd_angstrom']:11.4f} {row['closest_heavy_rmsd_angstrom']:13.4f} {row['first_vina_score_kcal_mol']:11.3f}")
for variant in dict.fromkeys(row['variant'] for row in record['comparison_rows']):
    rows = [row for row in record['comparison_rows'] if row['variant'] == variant]
    assert len(rows) == 6
    print(variant, 'first:', sum(row['first_pose_recovered'] for row in rows), '/6; returned set:', sum(row['returned_set_recovered'] for row in rows), '/6')
print('Counts describe these observed cells; no recovery probability or policy ranking.')

variant                      effort seed poses first-heavy closest-heavy first-score
native                           1    7     3     12.3453       12.3453     -10.899
native                           1   42     1     12.3663       12.3663     -10.779
native                           1 2026     3     12.3183       12.1256     -10.888
native                           8    7     3      0.2928        0.2928     -13.302
native                           8   42     5     12.3349       12.0659     -10.827
native                           8 2026     5     12.3183       12.0296     -10.888
published                        1    7     1      0.2718        0.2718     -13.293
published                        1   42     1      0.8969        0.8969     -13.286
published                        1 2026     3     12.3188       12.1504     -10.945
published                        8    7     4      0.9045        0.9045     -13.227
published                        8   42     4      0.8969        0.8969    

In [3]:
run = record['new_runs'][0]
restored = dmt.DockingResult.from_dict(run['result'])
assert dmt.verify_captured_inputs(restored.provenance['backend_artifacts'])
xyz = puw.get_value(puw.quantity(record['audit']['source_snapshot']['structures']['coordinates'], 'nm'), to_unit='angstrom')[0]
reference = puw.quantity(xyz[run['pdbqt_to_source_atom_indices']], 'angstrom')
with puw.context(standard_units=['pm', 'fs', 'K', 'mole', 'dalton', 'e', 'kJ/mol', 'radians']):
    report = dmt.evaluate_redocking(restored, reference, rmsd_cutoff=puw.quantity(250, 'pm'))
assert report['criterion'] == run['evaluation']['criterion']
np.testing.assert_allclose([row['rmsd'] for row in report['poses']], [row['rmsd'] for row in run['evaluation']['poses']], rtol=0, atol=1e-12)
print('Saved 40-atom evaluation agrees under pm/fs; no searches rerun.')

Saved 40-atom evaluation agrees under pm/fs; no searches rerun.
